# Chuuchuu punctuality pipeline (clean, consolidated)

A single, linear pipeline covering everything notebooks 1-5 did separately:

0. **Load raw data** -- CSV -> parquet (converted once, stored on the raw data drive folder)
1. **Country attribution** -- resolve a `country` for every stop
2. **Terminus identification** -- label each stop `depart` / `intermediate` / `terminus`, build `journey_id`, flag ambiguous/duplicate trips, classify `journey_type` (domestic/international)
3. **Operator identification** -- resolve a `normalized_operator` for every stop
4. **Cancellation status resolution** -- resolve `arrivalCancelled_resolved` / `departureCancelled_resolved`
5. **Per-train summary & export** -- collapse to one row per train, aggregate by year/operator/route type, export to `summary_stats/`

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

## Step 0 -- Dataset selection & raw data loading

Pick which raw extract to work on, then load it. The raw CSVs live on the shared drive (too large for the repo); we keep a parquet mirror right next to each CSV so re-running this notebook doesn't re-parse a multi-GB CSV every time.

**No local cache is used** -- earlier versions of this pipeline cached a copy of the raw import under `raw_data/cache/` in the repo, one file per `data_selection`. That's gone: the parquet mirror below lives on the shared drive alongside its source CSV instead (so it survives across machines and doesn't bloat the local repo checkout), and it's a straight mirror of the CSV -- not a per-selection, per-machine cache.

In [2]:
data_selection = "french"  # "french" | "combined" | "june" | "february"

# personal PC / VM shared-drive mount points for the Chuuchuu folder -- add a new entry here if
# this pipeline runs somewhere else and the folder isn't found
candidate_chuuchuu_folders = [
    r"G:\.shortcut-targets-by-id\1fop_2EKRGMK369J15pyuC9039CxT_tnw\Rail_databases\punctuality_big_data\Chuuchuu",  # personal PC
    r"D:\.shortcut-targets-by-id\18oT0tHU4ycBoKA7-PZZc_W0DyRedU_Jt\T&A\Shared knowledge tools and data\data_sources\Rail_databases\punctuality_big_data\Chuuchuu",  # VM
]

data_chuuchuu_folder = next((path for path in candidate_chuuchuu_folders if os.path.isdir(path)), None)
if data_chuuchuu_folder is None:
    raise FileNotFoundError(
        "Could not find the Chuuchuu shared-drive folder on this machine -- no path in "
        "candidate_chuuchuu_folders exists. Add this machine's local mount path to the list above."
    )
print(f"Using Chuuchuu folder: {data_chuuchuu_folder}")

Using Chuuchuu folder: D:\.shortcut-targets-by-id\18oT0tHU4ycBoKA7-PZZc_W0DyRedU_Jt\T&A\Shared knowledge tools and data\data_sources\Rail_databases\punctuality_big_data\Chuuchuu


In [3]:
raw_csv_filename_by_selection = {
    "combined": "delay_records_test_combined_EU.csv",
    "june": "delay_records_2026-06-10_to_2026-06-17_EU.csv",
    "february": "delay_records_2026-02-26_to_2026-03-04_EU.csv",
    "french": "delay_records_FR_2025.csv",
}
raw_csv_filename = raw_csv_filename_by_selection[data_selection]
raw_csv_path = f"{data_chuuchuu_folder}/{raw_csv_filename}"
raw_parquet_path = f"{data_chuuchuu_folder}/{os.path.splitext(raw_csv_filename)[0]}.parquet"

try:
    data_raw = pd.read_parquet(raw_parquet_path)
    print(f"Loaded existing parquet mirror: {raw_parquet_path}")
except FileNotFoundError:
    print(f"No parquet mirror at {raw_parquet_path} yet -- converting from CSV (slow, one-off)")
    data_raw = pd.read_csv(raw_csv_path, low_memory=False)
    data_raw.to_parquet(raw_parquet_path)
    data_raw = pd.read_parquet(raw_parquet_path)
    print(f"Saved parquet mirror to {raw_parquet_path}")

data_raw.shape

Loaded existing parquet mirror: D:\.shortcut-targets-by-id\18oT0tHU4ycBoKA7-PZZc_W0DyRedU_Jt\T&A\Shared knowledge tools and data\data_sources\Rail_databases\punctuality_big_data\Chuuchuu/delay_records_FR_2025.parquet


(7038209, 23)

In [4]:
# keep data_raw untouched as a record of the original import; this pipeline works on this copy instead
data_chuuchuu = data_raw.copy()

## Step 1 -- Country attribution

Identify the country of every stop using `deutscheBahnStopId` (UIC codes), with a series of fallbacks for the ids that don't resolve that way.

### Identify countries using deutscheBahnStopId (UIC codes)

The first two digits of `deutscheBahnStopId` are the UIC country code. We map them to a country using `sup_data/UIC_country_codes.csv`.

If `deutscheBahnStopId` is not exactly 7 characters, it isn't a valid UIC-based stop id, so it's flagged with an error value instead of guessing a country.

In [5]:
uic_codes = pd.read_csv("sup_data/UIC_country_codes.csv", sep=";")
uic_codes["Numerical code"] = uic_codes["Numerical code"].astype(str)

data_chuuchuu["deutscheBahnStopId"] = data_chuuchuu["deutscheBahnStopId"].astype(str)

# a valid UIC-based stop id is exactly 7 characters long; anything else signals a data issue
invalid_stop_id = data_chuuchuu["deutscheBahnStopId"].str.len() != 7
print(f"{invalid_stop_id.sum()} rows have a deutscheBahnStopId that is not exactly 7 characters")

data_chuuchuu["uicCodeStop"] = data_chuuchuu["deutscheBahnStopId"].str[:2]

0 rows have a deutscheBahnStopId that is not exactly 7 characters


In [6]:
data_chuuchuu = data_chuuchuu.merge(
    uic_codes[["Numerical code", "Country"]],
    how="left",                # keep all rows in data_chuuchuu
    left_on="uicCodeStop",     # column in data_chuuchuu
    right_on="Numerical code"  # column in uic_codes
)
data_chuuchuu = data_chuuchuu.drop(columns=["Numerical code"])
data_chuuchuu = data_chuuchuu.rename(columns={"Country": "country"})

# overwrite the country for stop ids we flagged as invalid, instead of trusting a coincidental match
data_chuuchuu.loc[invalid_stop_id, "country"] = "ERROR_INVALID_STOP_ID"

print(f"{len(data_chuuchuu['country'].unique())} distinct country values after the UIC-prefix match")
data_chuuchuu["country"].value_counts(dropna=False)

9 distinct country values after the UIC-prefix match


country
France            6517846
Switzerland        243158
Germany            105704
Belgium             72131
Netherlands         34650
Luxembourg          29047
United Kingdom      18370
Italy               15098
Spain                2205
Name: count, dtype: int64

### Rule-based fixes for `ERROR_INVALID_STOP_ID` / NaN countries

A few agencies have well-defined, agency-specific id quirks (verified by name-matching `stopName` against `sup_data/stations.csv`, which gave the same country nearly 100% of the time within each group):

- **`GTFSDE`, `NS`, `DB`** sometimes report a stop id with only 6 characters instead of the usual 7 (missing one digit) -- these resolve to **Germany**.
- **`PL`** sometimes reports ids in other formats (5, 8, 9 or 10 characters, e.g. the composite `"44420_1_1"` style) -- these resolve to **Poland**.
- **`HU`** uses two internal numbering prefixes, `36` (HÉV suburban lines) and `43` (GySEV/Raaberbahn regional lines), which aren't official UIC country codes -- these resolve to **Hungary**.

These rules only apply to rows currently marked `ERROR_INVALID_STOP_ID` or NaN -- they never override a country already found via the UIC prefix match.

In [7]:
# rows we're still allowed to touch: only the ones without a country yet
needs_fix = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{needs_fix.sum()} rows still need a country before the rule-based fixes")

id_length = data_chuuchuu["deutscheBahnStopId"].str.len()

# GTFSDE / NS / DB report some stops with a 6-character id (missing the usual 7th digit) -> German stops
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "GTFSDE") & (id_length == 6), "country"] = "Germany"
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "NS") & (id_length == 6), "country"] = "Germany"
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "DB") & (id_length == 6), "country"] = "Germany"

# PL reports ids in non-standard formats (5, 8, 9 or 10 characters, e.g. "44420_1_1") -> Polish stops
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "PL") & (id_length != 7), "country"] = "Poland"

# HU numbers some stops with internal prefixes 36 (HEV suburban lines) and 43 (GySEV/Raaberbahn
# regional lines), which aren't official UIC country codes -> Hungarian stops
uic_prefix = data_chuuchuu["deutscheBahnStopId"].str[:2]
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "HU") & (uic_prefix.isin(["36", "43"])), "country"] = "Hungary"

still_missing = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{still_missing.sum()} rows still need a country after the rule-based fixes")

0 rows still need a country before the rule-based fixes
0 rows still need a country after the rule-based fixes


### Fallback: name-based match against `stations.csv`

For any row still missing a country, fall back to matching `stopName` against `sup_data/stations.csv`'s `slug` column, which carries its own `country` field. This only fills in rows that are still `ERROR_INVALID_STOP_ID` or NaN -- it never overrides a country already resolved.

`stations.csv` stores countries as 2-letter codes (e.g. `"DE"`), so we translate them through `uic_codes` to keep `country` consistently made of full country names.

In [8]:
data_stations = pd.read_csv("sup_data/stations.csv", sep=";", low_memory=False)
stations_unique = data_stations.drop_duplicates(subset="slug", keep="first").copy()


def slugify(series):
    return (
        series.str.normalize("NFKD")     # split accented letters from their accents
        .str.encode("ascii", "ignore")   # drop the accents
        .str.decode("utf-8")
        .str.lower()
        .str.strip()
        .str.replace(r"\(", "-", regex=True)
        .str.replace(r"\)", "", regex=True)
        .str.replace(r"[^a-z0-9]+", "-", regex=True)  # collapse remaining punctuation/spaces
        .str.strip("-")
    )


data_chuuchuu["stopName_slug"] = slugify(data_chuuchuu["stopName"])

# translate stations.csv's 2-letter country codes into the full names used in our country column
alpha_to_country = uic_codes.drop_duplicates(subset="Alphabetical code", keep="first").set_index("Alphabetical code")["Country"]
stations_unique["country_full"] = stations_unique["country"].map(alpha_to_country)
slug_to_country = stations_unique.set_index("slug")["country_full"]

data_chuuchuu.loc[still_missing, "country"] = data_chuuchuu.loc[still_missing, "stopName_slug"].map(slug_to_country)

still_missing_after_fallback = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{still_missing_after_fallback.sum()} rows still need a country after the name-based fallback")

0 rows still need a country after the name-based fallback


### Known exceptions

Cases where the id-based logic gives the wrong country (or no country at all) regardless of the rules above, each individually verified by web search and recorded in `sup_data/known_exceptions_country_match.csv`:

- **Basel Bad Bf** is physically in Switzerland, but is managed by DB and numbered as if it were a German station (`80` prefix) -- should be **Switzerland**.
- **Charles de Gaulle Airport** has a `deutscheBahnStopId` whose `99` prefix coincidentally matches Iraq -- should be **France**.
- **41 OEBB stops** (Vienna-area S-Bahn/bus stops, plus regional stops in Tyrol, Styria, Carinthia and Vorarlberg) that carry a platform/street annotation `stations.csv` doesn't have a matching slug for -- should be **Austria**, except **Mittenwald**, a real town in Bavaria, Germany, even though it sits on an OEBB-served route through Tyrol.
- **PL**'s internal stop numbering doesn't always follow UIC conventions, and several of its ids coincidentally collide with ids that other agencies (GTFSDE, FR, SBB, NMBS) use as real UIC-prefixed ids for a completely different physical stop (e.g. `8012650` is PL's Pleszew *and* GTFSDE's Plessa). A handful of these PL stops are recorded here -- should be **Poland**.
- **IT's Statte** name-matched to the Belgian town of Statte (near Huy, Wallonia) via the `stations.csv` fallback -- should be **Italy**. This stop has no `deutscheBahnStopId` at all, so it can't be keyed the same way as the exceptions above.

We join on **both `agency` and `deutscheBahnStopId` together**, not the id alone -- otherwise a PL exception would leak onto an unrelated stop that just happens to share the same numeric id under a different agency. For exceptions with no `deutscheBahnStopId` (like Statte), we instead join on `agency` + `originalStopId`, which is unique per physical stop. Either way, the exception always wins regardless of what the automated rules produced for that stop.

In [9]:
known_exceptions = pd.read_csv(
    "sup_data/known_exceptions_country_match.csv",
    dtype={"deutscheBahnStopId": str, "originalStopId": str},
)

# join on (agency, deutscheBahnStopId) together, not just the id: the same numeric id can coincidentally
# belong to two unrelated physical stops under different agencies, so an id-only join would leak one
# agency's exception onto a different, unrelated stop under another agency
id_exceptions = known_exceptions[known_exceptions["deutscheBahnStopId"].notna()]
exception_key = id_exceptions["agency"] + "|" + id_exceptions["deutscheBahnStopId"]
exception_country = pd.Series(id_exceptions["country"].values, index=exception_key)

data_key = data_chuuchuu["agency"] + "|" + data_chuuchuu["deutscheBahnStopId"]
data_chuuchuu["country"] = data_key.map(exception_country).fillna(data_chuuchuu["country"])

# exceptions with no deutscheBahnStopId (e.g. IT's Statte) can't be keyed the same way: since the id
# column is stringified to the literal "nan" for missing values, an id-based join would match every
# other stop under that agency that's also missing an id. These instead join on (agency, originalStopId)
originalid_exceptions = known_exceptions[known_exceptions["originalStopId"].notna()]
exception_key_orig = originalid_exceptions["agency"] + "|" + originalid_exceptions["originalStopId"]
exception_country_orig = pd.Series(originalid_exceptions["country"].values, index=exception_key_orig)

data_key_orig = data_chuuchuu["agency"] + "|" + data_chuuchuu["originalStopId"].astype(str)
data_chuuchuu["country"] = data_key_orig.map(exception_country_orig).fillna(data_chuuchuu["country"])

### SBB stops: join against the official Swiss stop register (Didok)

SBB reports many stops using Swiss "sloid" identifiers (e.g. `ch:1:sloid:2204`) instead of a UIC-prefixed code -- these are small/regional stops that aren't in `stations.csv`, so the name-based fallback above can't match them.

The Federal Office of Transport's official stop register (`sup_data/switzerland_stations_didok.csv`) uses the exact same `sloid` identifiers and includes an ISO country code per stop, so we join directly on the id instead of matching by name. This only touches SBB rows still unresolved after the fallback above.

In [10]:
swiss_stations = pd.read_csv("sup_data/switzerland_stations_didok.csv", sep=";", low_memory=False)

# Liechtenstein ("LI") has no UIC country code of its own, so it's missing from uic_codes -> add it manually
alpha_to_country_extra = pd.concat([alpha_to_country, pd.Series({"LI": "Liechtenstein"})])

sloid_to_country = (
    swiss_stations.drop_duplicates(subset="sloid")
    .set_index("sloid")["isocountrycode"]
    .map(alpha_to_country_extra)
)

sbb_still_missing = still_missing_after_fallback & (data_chuuchuu["agency"] == "SBB")
print(f"{sbb_still_missing.sum()} SBB rows targeted for the Didok join")

data_chuuchuu.loc[sbb_still_missing, "country"] = data_chuuchuu.loc[sbb_still_missing, "deutscheBahnStopId"].map(sloid_to_country)

still_missing_after_fallback = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{still_missing_after_fallback.sum()} rows still need a country after the SBB Didok join")

0 SBB rows targeted for the Didok join
0 rows still need a country after the SBB Didok join


### OEBB stops: retry the name match after stripping bus-platform annotations

Many OEBB stop names carry a platform/street annotation in parentheses, e.g. `"Bruck/Leitha Bahnhof (Bahnhofplatz)"`. The fallback's `slugify()` only replaces `(`/`)` with `-`/nothing -- it doesn't remove the annotation text itself, so the slug never matches `stations.csv`'s plain version.

This retries the match for OEBB rows still unresolved, using a slug that strips the whole parenthetical suffix instead of just the brackets.

In [11]:
def slugify_stripped(series):
    return (
        series.str.normalize("NFKD")
        .str.encode("ascii", "ignore")
        .str.decode("utf-8")
        .str.lower()
        .str.strip()
        .str.replace(r"\(.*?\)", "", regex=True)   # drop the whole parenthetical suffix, not just the brackets
        .str.replace(r"[^a-z0-9]+", "-", regex=True)
        .str.strip("-")
    )


oebb_still_missing = still_missing_after_fallback & (data_chuuchuu["agency"] == "OEBB")
print(f"{oebb_still_missing.sum()} OEBB rows targeted for the retry")

retry_slug = slugify_stripped(data_chuuchuu.loc[oebb_still_missing, "stopName"])
data_chuuchuu.loc[oebb_still_missing, "country"] = retry_slug.map(slug_to_country)

still_missing_after_fallback = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{still_missing_after_fallback.sum()} rows still need a country after the OEBB retry")

0 OEBB rows targeted for the retry
0 rows still need a country after the OEBB retry


In [12]:
# anything left is genuinely unresolved -> label it explicitly instead of leaving NaN / the old error value
data_chuuchuu.loc[still_missing_after_fallback, "country"] = "UNKNOWN_COUNTRY"

print(f"{len(data_chuuchuu['country'].unique())} distinct country values")
display(data_chuuchuu["country"].value_counts(dropna=False))

pct_unknown = len(data_chuuchuu[data_chuuchuu["country"] == "UNKNOWN_COUNTRY"]) / len(data_chuuchuu) * 100
print(f"\n{pct_unknown:.2f}% of rows have an unknown country")

9 distinct country values


country
France            6517846
Switzerland        243158
Germany            105704
Belgium             72131
Netherlands         34650
Luxembourg          29047
United Kingdom      18370
Italy               15098
Spain                2205
Name: count, dtype: int64


0.00% of rows have an unknown country


### Validation: flag every agency's minority countries for review

For every agency, treat its single most common country as its "home" country, and flag every other (agency, country) combination it has any rows under -- regardless of size -- as a candidate to manually verify, the same way SBB/OEBB/PL were investigated above. This doesn't auto-fix anything; it's a checklist. A flagged combination with plausible geography (e.g. `DB`/Germany -> Austria, a real neighboring country with heavy rail traffic) is likely fine as-is; one with implausible geography or a tiny row count is worth investigating the same way, with any confirmed error added to `sup_data/known_exceptions_country_match.csv`.

In [13]:
agency_country_counts = (
    data_chuuchuu.groupby(["agency", "country"])
    .agg(rows=("deutscheBahnStopId", "size"), unique_stops=("deutscheBahnStopId", "nunique"))
    .reset_index()
)

# use unique stops (not rows) as the basis for the percentage: a single busy stop racks up far more
# rows than a real error affecting several rarely-serviced stops, so row counts don't reflect how
# many distinct stations are potentially mislabeled
agency_stop_totals = agency_country_counts.groupby("agency")["unique_stops"].transform("sum")
agency_country_counts["share_pct"] = (agency_country_counts["unique_stops"] / agency_stop_totals * 100).round(2)

# each agency's single most common country (by unique stops) is treated as its "home" country
home_country = agency_country_counts.loc[agency_country_counts.groupby("agency")["unique_stops"].idxmax(), ["agency", "country"]]
home_country = home_country.set_index("agency")["country"]
agency_country_counts["is_home_country"] = agency_country_counts["country"] == agency_country_counts["agency"].map(home_country)

flagged_summary = agency_country_counts[~agency_country_counts["is_home_country"]].sort_values(["agency", "unique_stops"], ascending=[True, False])
print(f"{len(flagged_summary)} (agency, country) combinations flagged for manual review (not the agency's home country)")
with pd.option_context("display.max_rows", None):
    display(flagged_summary.drop(columns=["is_home_country"]))

8 (agency, country) combinations flagged for manual review (not the agency's home country)


,agency,country,rows,unique_stops,share_pct
2,FR,Germany,105704,77,2.03
7,FR,Switzerland,243158,72,1.90
3,FR,Italy,15098,24,0.63
0,FR,Belgium,72131,18,0.47
4,FR,Luxembourg,29047,9,0.24
6,FR,Spain,2205,4,0.11
5,FR,Netherlands,34650,3,0.08
8,FR,United Kingdom,18370,1,0.03


## Step 2 -- Terminus identification

For every row, identify whether it is the departure stop, an intermediate stop, or the terminus (final stop) of its trip -- new column `depart_terminus`.

Guidance from the data provider:

> Each row should have a unique (agency, routeType, routeNumber, date, deutscheBahnStopId) tuple. For a given date, there may be a few cases where the same (agency, routeType and routeNumber) will contain records of two different trains (specifically, overlap between Austria and Germany).

> For a given (agency, routeType, routeNumber, date) tuple I would order the records by "coalesce(arrival, departure, plannedArrival, plannedDeparture)". The first record is the departure point, and the last record is the arrival point. Of course, keeping [the overlap] in mind, this can sometimes produce incorrect results.

So a trip is identified by (`agency`, `routeType`, `routeNumber`, `date`) -- kept around as `journey_id`. Rows get labelled:
- `depart` -- first stop of the trip (by the coalesced timestamp)
- `terminus` -- last stop of the trip
- `intermediate` -- everything in between
- `unknown` -- whenever we can't be confident in the ordering (no timestamp at all to sort by, or the trip is one of the ambiguous "two trains, one route number" cases the provider warned about)

### Build the sort key: `coalesce(arrival, departure, plannedArrival, plannedDeparture)`

The first non-null value among the four timestamp columns, per row.

In [14]:
timestamp_cols = ["arrival", "departure", "plannedArrival", "plannedDeparture"]
for col in timestamp_cols:
    data_chuuchuu[col] = pd.to_datetime(data_chuuchuu[col], utc=True, errors="coerce")

# Use arrival if not NA, otherwise departure, otherwise plannedArrival, otherwise plannedDeparture
data_chuuchuu["sort_time"] = (
    data_chuuchuu["arrival"]
    .fillna(data_chuuchuu["departure"])
    .fillna(data_chuuchuu["plannedArrival"])
    .fillna(data_chuuchuu["plannedDeparture"])
)

# track which column was actually used
conditions = [
    data_chuuchuu["arrival"].notna(),
    data_chuuchuu["departure"].notna(),
    data_chuuchuu["plannedArrival"].notna(),
    data_chuuchuu["plannedDeparture"].notna(),
]
choices = ["arrival", "departure", "plannedArrival", "plannedDeparture"]
data_chuuchuu["sort_time_source"] = np.select(conditions, choices, default=None)

print(f"{data_chuuchuu['sort_time'].isna().sum()} rows have none of arrival/departure/plannedArrival/plannedDeparture -> can't be placed in the order")
data_chuuchuu["sort_time_source"].value_counts(dropna=False)

0 rows have none of arrival/departure/plannedArrival/plannedDeparture -> can't be placed in the order


sort_time_source
arrival             6169318
departure            858358
plannedArrival         8248
plannedDeparture       2285
Name: count, dtype: int64

### Flag trips affected by the "two trains, one route number" ambiguity

The provider said each (`agency`, `routeType`, `routeNumber`, `date`, `deutscheBahnStopId`) tuple should be unique. A repeated `deutscheBahnStopId` within the same (`agency`, `routeType`, `routeNumber`, `date`) trip is exactly a violation of that -- the same signal as the Austria/Germany overlap case the provider described. When that happens the ordering for the whole trip can't be trusted (we can't tell which duplicate row belongs to which physical train), so every row in that trip is marked `unknown` rather than guessed at.

In [15]:
data_chuuchuu["journey_id"] = (
    data_chuuchuu["agency"].astype(str) + "_" +
    data_chuuchuu["routeType"].astype(str) + "_" +
    data_chuuchuu["routeNumber"].astype(str) + "_" +
    data_chuuchuu["date"].astype(str)
)

dup_stop_in_trip = data_chuuchuu.duplicated(subset=["journey_id", "deutscheBahnStopId"], keep=False)
ambiguous_journey_ids = set(data_chuuchuu.loc[dup_stop_in_trip, "journey_id"])
print(f"{len(ambiguous_journey_ids)} trips contain a repeated stop id -> flagged as ambiguous")

data_chuuchuu["is_ambiguous_trip"] = data_chuuchuu["journey_id"].isin(ambiguous_journey_ids)
print(f"{data_chuuchuu['is_ambiguous_trip'].sum()} rows belong to an ambiguous trip")

1 trips contain a repeated stop id -> flagged as ambiguous
46 rows belong to an ambiguous trip


### Flag trains that may be duplicated across agencies

Separate concern from the ambiguity check above: that one catches a repeated stop *within* one agency's version of a trip. This one checks whether the same physical trip -- same `routeType`, `routeNumber`, `date`, `deutscheBahnStopId` -- shows up under more than one `agency`, by grouping on `journey_verificator` (deliberately excluding `agency`) and counting how many *distinct* agencies report it.

**Caveat before treating a flag as an error:** on the full EU dataset, the top flagged combinations are overwhelmingly neighboring-country agency pairs (DB/OEBB, DB/GTFSDE, GTFSDE/SBB, FR/SBB, DB/NS, ...) -- i.e. exactly the border pairs you'd expect if an international train is legitimately tracked by more than one national real-time feed. So a flag here isn't automatically a duplicate/error -- it's a checklist to review, refined further below by checking timestamp agreement.

In [16]:
data_chuuchuu["journey_verificator"] = (
    data_chuuchuu["routeType"].astype(str) + "_" +
    data_chuuchuu["routeNumber"].astype(str) + "_" +
    data_chuuchuu["date"].astype(str) + "_" +
    data_chuuchuu["deutscheBahnStopId"].astype(str)
)

# count distinct agencies reporting each (routeType, routeNumber, date, stop) combination
agencies_per_verificator = data_chuuchuu.groupby("journey_verificator")["agency"].nunique()
cross_agency_verificators = set(agencies_per_verificator[agencies_per_verificator > 1].index)
print(f"{len(cross_agency_verificators)} (routeType, routeNumber, date, stop) combinations are reported by more than one agency")

data_chuuchuu["is_cross_agency_duplicate"] = data_chuuchuu["journey_verificator"].isin(cross_agency_verificators)
print(f"{data_chuuchuu['is_cross_agency_duplicate'].sum()} rows belong to one of these combinations")

flagged = data_chuuchuu[data_chuuchuu["is_cross_agency_duplicate"]]
agency_groups = flagged.groupby("journey_verificator")["agency"].apply(lambda s: tuple(sorted(s.unique())))
agency_groups.value_counts().head(20)

0 (routeType, routeNumber, date, stop) combinations are reported by more than one agency
0 rows belong to one of these combinations


Series([], Name: count, dtype: int64)

### How confident can we be that a cross-agency match is a real duplicate?

If two agencies are reporting the *same physical stop event*, their `sort_time` should land within a couple of minutes of each other. If instead it's a coincidental collision -- two unrelated trains that happen to reuse the same `routeNumber` at the same station on the same day under different agencies -- there's no reason for their times to line up; the gap should look essentially random.

So: for every flagged `journey_verificator`, compute the spread (max - min) of `sort_time` across its rows. A small spread is strong evidence of a genuine duplicate; a large spread points to a coincidental collision instead.

In [17]:
if flagged.empty:
    # no cross-agency duplicates flagged at all (e.g. a single-agency dataset) -- groupby-agg on an
    # empty frame would otherwise inherit sort_time's datetime dtype instead of float, breaking the
    # <= comparison below
    time_spread_minutes = pd.Series(dtype="float64")
else:
    time_spread_minutes = (
        flagged.groupby("journey_verificator")["sort_time"]
        .agg(lambda s: (s.max() - s.min()).total_seconds() / 60 if s.notna().sum() >= 2 else np.nan)
    )

print(time_spread_minutes.describe())
print()
print(f"{(time_spread_minutes <= 15).mean() * 100:.1f}% of flagged combinations have every agency reporting within 15 minutes of each other")
print(f"{(time_spread_minutes > 60).mean() * 100:.1f}% have a gap of more than an hour -- likely a coincidental routeNumber collision, not a true duplicate")
print(f"{time_spread_minutes.isna().mean() * 100:.1f}% can't be checked this way (a sort_time is missing on at least one side)")

TIME_SPREAD_THRESHOLD_MINUTES = 15
likely_true_duplicate = set(time_spread_minutes[time_spread_minutes <= TIME_SPREAD_THRESHOLD_MINUTES].index)

data_chuuchuu["cross_agency_duplicate_confidence"] = "not_flagged"
data_chuuchuu.loc[data_chuuchuu["is_cross_agency_duplicate"], "cross_agency_duplicate_confidence"] = "needs_review"
data_chuuchuu.loc[data_chuuchuu["journey_verificator"].isin(likely_true_duplicate), "cross_agency_duplicate_confidence"] = "likely_true_duplicate"

data_chuuchuu["cross_agency_duplicate_confidence"].value_counts()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
dtype: float64

nan% of flagged combinations have every agency reporting within 15 minutes of each other
nan% have a gap of more than an hour -- likely a coincidental routeNumber collision, not a true duplicate
nan% can't be checked this way (a sort_time is missing on at least one side)


cross_agency_duplicate_confidence
not_flagged    7038209
Name: count, dtype: int64

### Assign `depart` / `intermediate` / `terminus` by rank of `sort_time` within each trip

For rows in a non-ambiguous trip that do have a `sort_time`: rank them within their `journey_id`. Rank 1 is `depart`, the highest rank is `terminus`, everything else is `intermediate`.

A trip with only one row that has a usable `sort_time` is both "first" and "last" at once -- we can't tell whether that's genuinely a single-stop trip or whether the true departure/terminus simply has no timestamp, so it's left as `unknown` rather than guessing.

Everything else -- no `sort_time` at all, or part of an ambiguous trip -- stays `unknown` (the column's default).

In [18]:
data_chuuchuu["depart_terminus"] = "unknown"

usable = (~data_chuuchuu["is_ambiguous_trip"]) & data_chuuchuu["sort_time"].notna()

grouped_sort_time = data_chuuchuu.groupby("journey_id", sort=False)["sort_time"]
rank = grouped_sort_time.rank(method="first")
usable_count_per_trip = grouped_sort_time.transform("count")  # count() ignores NaT, i.e. only usable rows

is_first = usable & (rank == 1)
is_last = usable & (rank == usable_count_per_trip)

data_chuuchuu.loc[is_first & ~is_last, "depart_terminus"] = "depart"
data_chuuchuu.loc[is_last & ~is_first, "depart_terminus"] = "terminus"
data_chuuchuu.loc[usable & ~is_first & ~is_last, "depart_terminus"] = "intermediate"

print(data_chuuchuu["depart_terminus"].value_counts(dropna=False))
print()
print(round((data_chuuchuu["depart_terminus"] == "unknown").mean() * 100, 2), "% of rows have unknown depart_terminus")

depart_terminus
intermediate    5322581
depart           857791
terminus         857791
unknown              46
Name: count, dtype: int64

0.0 % of rows have unknown depart_terminus


### Invariant checks

These test the labeling *logic* itself (not the underlying data) -- they should always pass regardless of what the raw data looks like, so a failure here means a bug in the code above, not a data quality issue:

1. Every resolved journey has exactly one `depart` row and exactly one `terminus` row.
2. The `depart` row's `sort_time` is the minimum, and the `terminus` row's `sort_time` is the maximum, within its journey.
3. No row is labeled `depart`/`intermediate`/`terminus` if it has no `sort_time` or belongs to an ambiguous trip -- those must always stay `unknown`.

In [19]:
# work off a slim 4-column copy for these checks -- filtering/grouping the full frame repeatedly
# is needlessly memory-heavy given how many object-dtype columns it carries
check_cols = data_chuuchuu[["journey_id", "depart_terminus", "sort_time", "is_ambiguous_trip"]]

resolved = check_cols[check_cols["depart_terminus"] != "unknown"]

# 1. exactly one depart and one terminus per resolved journey
counts_per_journey = resolved.groupby("journey_id")["depart_terminus"].value_counts().unstack(fill_value=0)
bad_depart_count = counts_per_journey[counts_per_journey.get("depart", 0) != 1]
bad_terminus_count = counts_per_journey[counts_per_journey.get("terminus", 0) != 1]
print(f"journeys with != 1 depart row: {len(bad_depart_count)}")
print(f"journeys with != 1 terminus row: {len(bad_terminus_count)}")
assert len(bad_depart_count) == 0, "found a resolved journey without exactly one depart row"
assert len(bad_terminus_count) == 0, "found a resolved journey without exactly one terminus row"

# 2. depart is the min sort_time, terminus is the max sort_time, within each resolved journey
journey_time_bounds = resolved.groupby("journey_id")["sort_time"].agg(["min", "max"])
depart_rows = resolved[resolved["depart_terminus"] == "depart"].set_index("journey_id")
terminus_rows = resolved[resolved["depart_terminus"] == "terminus"].set_index("journey_id")

depart_not_min = depart_rows["sort_time"] != journey_time_bounds.loc[depart_rows.index, "min"]
terminus_not_max = terminus_rows["sort_time"] != journey_time_bounds.loc[terminus_rows.index, "max"]
print(f"depart rows NOT at the min sort_time of their journey: {depart_not_min.sum()}")
print(f"terminus rows NOT at the max sort_time of their journey: {terminus_not_max.sum()}")
assert depart_not_min.sum() == 0
assert terminus_not_max.sum() == 0

# 3. a resolved label should never occur for a row with no sort_time or in an ambiguous trip
should_be_unknown = check_cols["sort_time"].isna() | check_cols["is_ambiguous_trip"]
mislabeled = check_cols[(check_cols["depart_terminus"] != "unknown") & should_be_unknown]
print(f"rows labeled non-unknown despite no sort_time or an ambiguous trip: {len(mislabeled)}")
assert len(mislabeled) == 0

del check_cols, resolved
print("all invariant checks passed")

journeys with != 1 depart row: 0
journeys with != 1 terminus row: 0
depart rows NOT at the min sort_time of their journey: 0
terminus rows NOT at the max sort_time of their journey: 0
rows labeled non-unknown despite no sort_time or an ambiguous trip: 0
all invariant checks passed


### Domestic vs. international journeys

For a given `journey_id`, compare the `country` of its `depart` row against the `country` of its `terminus` row: if they differ, the service is `international`; if they match, it's `domestic`.

This only works for journeys where both endpoints were confidently resolved above. A journey stays `unknown` if it's `is_ambiguous_trip`, if either endpoint has no `sort_time` to rank by, or if the `country` at either endpoint itself couldn't be resolved (e.g. `UNKNOWN_COUNTRY`/NaN) -- a missing country should never silently read as "differs from" a known one and get mislabeled `international`.

In [20]:
depart_country = data_chuuchuu.loc[data_chuuchuu["depart_terminus"] == "depart"].set_index("journey_id")["country"]
terminus_country = data_chuuchuu.loc[data_chuuchuu["depart_terminus"] == "terminus"].set_index("journey_id")["country"]

# treat UNKNOWN_COUNTRY the same as a missing country -- neither side should count as "known" for this comparison
depart_country = depart_country.replace("UNKNOWN_COUNTRY", np.nan)
terminus_country = terminus_country.replace("UNKNOWN_COUNTRY", np.nan)

journey_type = pd.Series("unknown", index=pd.Index(data_chuuchuu["journey_id"].unique(), name="journey_id"))

comparable = depart_country.index.intersection(terminus_country.index)
both_known = depart_country.loc[comparable].notna() & terminus_country.loc[comparable].notna()
comparable = comparable[both_known]

is_international = depart_country.loc[comparable] != terminus_country.loc[comparable]
journey_type.loc[comparable] = np.where(is_international, "international", "domestic")

data_chuuchuu["journey_type"] = data_chuuchuu["journey_id"].map(journey_type)

print(data_chuuchuu.drop_duplicates("journey_id")["journey_type"].value_counts(dropna=False))

journey_type
domestic         728379
international    129412
unknown               1
Name: count, dtype: int64


## Step 3 -- Operator identification

Identify the rail operator for each row, translating the SQL rules from `instructions_chuuchuu_operator_identification.md` into pandas:

- `UPPER(TRIM(col))` comparisons -> normalized `_norm` columns.
- `routeNumber` list/CAST(...AS INTEGER) BETWEEN comparisons -> a trimmed string column plus a numeric column (non-numeric values become NaN and simply won't match any range check).
- `stopcountry = 'XX'` -> the full country name already resolved in step 1's `country` column.

Rules are applied in the same order as the instructions doc, so later rules intentionally overwrite `operator` for rows also matched by an earlier, broader rule -- exactly like running the SQL `UPDATE`s in sequence.

In [21]:
# some data_selection subsets (e.g. "french") have no operator info at all, so pandas infers
# an all-NaN float64 dtype for this column on load -- cast to object so the string assignments
# below don't fail with a dtype mismatch
data_chuuchuu["operator"] = data_chuuchuu["operator"].astype("object")


def norm(series):
    """Mirror SQL's UPPER(TRIM(...)) comparison."""
    return series.astype(str).str.strip().str.upper()


def in_ranges(int_series, ranges):
    """OR together several inclusive (lo, hi) BETWEEN checks; NaN (non-numeric routeNumber) never matches."""
    mask = pd.Series(False, index=int_series.index)
    for lo, hi in ranges:
        mask |= int_series.between(lo, hi)
    return mask


data_chuuchuu["routeType_norm"] = norm(data_chuuchuu["routeType"])
data_chuuchuu["agency_norm"] = norm(data_chuuchuu["agency"])
data_chuuchuu["routeNumber_trim"] = data_chuuchuu["routeNumber"].astype(str).str.strip()
data_chuuchuu["routeNumber_int"] = pd.to_numeric(data_chuuchuu["routeNumber_trim"], errors="coerce")

print(f"{data_chuuchuu['routeNumber_int'].isna().sum()} rows have a non-numeric routeNumber (can't be matched by any routeNumber-range rule below)")

0 rows have a non-numeric routeNumber (can't be matched by any routeNumber-range rule below)


### Highspeed services

In [22]:
# routeType matched with its exact (case-sensitive) literal, same as the source SQL
highspeed_operator_by_routeType = {
    "ICE": "Deutsche Bahn",
    "TGV": "SNCF", "TGV INOUI": "SNCF", "TGV Lyria": "SNCF", "LYR": "SNCF", "LYRIA": "SNCF", "Ouigo": "SNCF", "OUI": "SNCF",
    "RJX": "OEBB", "rjx": "OEBB",
    "FR": "Trenitalia",
    "EST": "Eurostar", "EUR": "Eurostar", "Eurostar": "Eurostar",
    "Italo": "Italo",
}

highspeed_mask = data_chuuchuu["routeType"].isin(highspeed_operator_by_routeType)
data_chuuchuu.loc[highspeed_mask, "operator"] = data_chuuchuu.loc[highspeed_mask, "routeType"].map(highspeed_operator_by_routeType)
print(f"{highspeed_mask.sum()} rows assigned an operator via the highspeed routeType rule")

1215698 rows assigned an operator via the highspeed routeType rule


### Night train services

`NJ`/`Nightjet` -> OEBB, `ES`/`European Sleeper` -> European Sleeper, and `EN` split by `routeNumber` (a handful of `EN` route numbers are first relabeled to `NJ`, per the instructions).

In [23]:
# relabel the NJ-under-EN route numbers first, so the NJ operator rule below picks them up too
en_to_nj_route_numbers = {"294", "295", "13485", "233"}
en_to_nj_mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(en_to_nj_route_numbers)
data_chuuchuu.loc[en_to_nj_mask, "routeType"] = "NJ"
data_chuuchuu.loc[en_to_nj_mask, "routeType_norm"] = "NJ"
print(f"{en_to_nj_mask.sum()} rows had routeType 'EN' relabeled to 'NJ'")

data_chuuchuu.loc[data_chuuchuu["routeType_norm"].isin(["ES", "EUROPEAN SLEEPER"]), "operator"] = "European Sleeper"
data_chuuchuu.loc[data_chuuchuu["routeType_norm"].isin(["NJ", "NIGHTJET"]), "operator"] = "OEBB"

en_operator_by_routeNumber = {
    "SJ": {"344", "345", "346", "13471", "13472"},
    "HZ": {"40465", "40414", "40237", "414", "415"},
    "PKP": {"406", "407", "40417", "40416", "40407", "1276", "1277"},
    "Ceske Drahy": {"40458", "40459", "443", "442"},
    "MAV": {"40462", "40467", "50237", "50462", "40476", "40457", "40406", "462", "476", "477", "463"},
}
for operator_name, route_numbers in en_operator_by_routeNumber.items():
    mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(route_numbers)
    data_chuuchuu.loc[mask, "operator"] = operator_name
    print(f"{mask.sum()} EN rows assigned operator '{operator_name}'")

# route numbers the instructions explicitly flag as having no known operator yet -- left untouched
en_unknown_route_numbers = {
    "1415", "1153", "1152", "50476", "323",
    "13400", "13403", "13451", "13408", "13401", "13402", "13404", "13406", "13420", "13405", "13409", "13417",
    "93701", "319", "580", "230", "320", "34834", "91641", "91505", "32",
    "277", "37501", "89843", "89962", "11799", "28565", "28960", "370", "20159",
}
en_unknown_mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(en_unknown_route_numbers)
print(f"{en_unknown_mask.sum()} EN rows have no known operator per the instructions -- left as-is")

0 rows had routeType 'EN' relabeled to 'NJ'
0 EN rows assigned operator 'SJ'
0 EN rows assigned operator 'HZ'
0 EN rows assigned operator 'PKP'
0 EN rows assigned operator 'Ceske Drahy'
0 EN rows assigned operator 'MAV'
0 EN rows have no known operator per the instructions -- left as-is


### Railjet (OEBB or Ceske Drahy)

In [24]:
ceske_drahy_rj_route_numbers = {
    "50", "51", "52", "53", "54", "55", "56",
    "70", "71", "72", "73", "74", "75", "78", "79",
    "170", "171", "172", "173", "174", "175", "176", "177", "178", "179",
    "244", "250", "251", "252", "253", "254", "255", "256", "257", "258", "259",
    "270", "271", "272", "273", "274", "275", "276", "277",
    "284", "285",
    "370", "371", "372", "373", "374", "375",
    "382", "383", "384", "385", "386", "387",
    "478", "479",
    "512", "515",
    "548", "549",
    "576", "577", "578", "579",
    "644", "645",
}

rj_mask = data_chuuchuu["routeType_norm"] == "RJ"
is_ceske_drahy_rj = rj_mask & data_chuuchuu["routeNumber_trim"].isin(ceske_drahy_rj_route_numbers)
data_chuuchuu.loc[is_ceske_drahy_rj, "operator"] = "Ceske Drahy"
data_chuuchuu.loc[rj_mask & ~is_ceske_drahy_rj, "operator"] = "OEBB"
print(f"{is_ceske_drahy_rj.sum()} RJ rows assigned 'Ceske Drahy', {(rj_mask & ~is_ceske_drahy_rj).sum()} assigned 'OEBB'")

0 RJ rows assigned 'Ceske Drahy', 0 assigned 'OEBB'


### Easy operator cases (PL, FLX, HU, IT, FR)

In [25]:
pl_mask = data_chuuchuu["agency"] == "PL"
data_chuuchuu.loc[pl_mask, "operator"] = "PKP Intercity"

flx_mask = data_chuuchuu["routeType_norm"] == "FLX"
data_chuuchuu.loc[flx_mask, "operator"] = "Flixtrain"

hu_routeTypes = {"EC", "IC", "EX", "EN", "G", "GY", "ER", "H", "IR", "S", "SZ", "Z"}
hu_mask = (data_chuuchuu["agency_norm"] == "HU") & data_chuuchuu["routeType_norm"].isin(hu_routeTypes)
data_chuuchuu.loc[hu_mask, "operator"] = "MAV"

it_routeTypes = {"FR", "FA", "FB", "IC", "ICN", "EXP", "IR", "REG", "MET", "EN", "NCL"}
it_mask = (data_chuuchuu["agency_norm"] == "IT") & data_chuuchuu["routeType_norm"].isin(it_routeTypes)
data_chuuchuu.loc[it_mask, "operator"] = "Trenitalia"

fr_routeTypes = {
    "IC", "ICN", "INTERCITES", "INTERCITES DE NUIT", "LYR", "LYRIA",
    "NAV", "NAVETTE", "OGO", "OUI", "OUIGO", "TER", "TGV INOUI", "TRAIN TER",
    # T&E observation (not from the data provider): rail-replacement coach service under
    # the TER brand, and tram-train -- both confirmed SNCF
    "CAR TER", "TRAMTRAIN",
}
fr_mask = (data_chuuchuu["agency_norm"] == "FR") & data_chuuchuu["routeType_norm"].isin(fr_routeTypes)
data_chuuchuu.loc[fr_mask, "operator"] = "SNCF"

print(f"PL: {pl_mask.sum()}, FLX: {flx_mask.sum()}, HU: {hu_mask.sum()}, IT: {it_mask.sum()}, FR: {fr_mask.sum()}")

PL: 0, FLX: 0, HU: 0, IT: 0, FR: 6423335


### SBB agency

`IC`/`EC` -> SBB whenever the stop is in Switzerland. `IR`/`R`/`RE`/`S`/`SN` are only *sometimes* SBB, split by `routeNumber` blocks. `PE`/`ICE`/`RB` have no rule.

In [26]:
# IC, EC inside Switzerland -> SBB
sbb_ic_ec_mask = data_chuuchuu["routeType"].isin(["IC", "EC"]) & (data_chuuchuu["country"] == "Switzerland")
data_chuuchuu.loc[sbb_ic_ec_mask, "operator"] = "SBB"
print(f"{sbb_ic_ec_mask.sum()} IC/EC rows in Switzerland assigned 'SBB'")

# IR
sbb_ir_route_numbers = {
    "1651", "1652", "1653", "1654", "1656", "1658", "1662", "1671", "1673", "1674", "1675", "1676", "1677", "1678",
    "1900", "1902", "1904", "1910", "1929", "1943", "1945",
    "2306", "2344", "2354", "2357", "2358", "2361", "2366", "2370", "2374", "2379", "2381", "2388", "2390", "2392", "2393", "2394",
    "3016", "3017", "3022", "3024", "3026", "3029", "3030", "3110",
    "746", "749",
}
sbb_ir_ranges = [(1702, 1843), (1956, 1995), (2055, 2194), (2252, 2292), (2456, 2493), (2503, 2543), (2562, 2599), (2610, 2662), (3251, 3293)]

sbb_ir_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "IR")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_ir_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_ir_ranges))
)
data_chuuchuu.loc[sbb_ir_mask, "operator"] = "SBB"
print(f"{sbb_ir_mask.sum()} SBB IR rows assigned 'SBB'")

0 IC/EC rows in Switzerland assigned 'SBB'
0 SBB IR rows assigned 'SBB'


In [27]:
# R
sbb_r_ranges = [
    (14402, 14693), (17400, 17497), (18940, 18999), (23000, 23535), (24004, 24999),
    (25850, 25864), (26101, 26370), (5607, 5994), (6006, 6758), (7014, 7461), (9901, 9940),
]
sbb_r_mask = (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "R") & in_ranges(data_chuuchuu["routeNumber_int"], sbb_r_ranges)
data_chuuchuu.loc[sbb_r_mask, "operator"] = "SBB"
print(f"{sbb_r_mask.sum()} SBB R rows assigned 'SBB'")

# RE
sbb_re_route_numbers = {"2087", "2089", "2090", "2092", "2560", "741", "742", "744"}
sbb_re_ranges = [(18122, 18495), (25500, 25843), (2600, 2607), (3564, 3685), (3958, 3995), (4706, 4716), (4762, 4842), (4908, 4940)]
sbb_re_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "RE")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_re_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_re_ranges))
)
data_chuuchuu.loc[sbb_re_mask, "operator"] = "SBB"
print(f"{sbb_re_mask.sum()} SBB RE rows assigned 'SBB'")

0 SBB R rows assigned 'SBB'
0 SBB RE rows assigned 'SBB'


In [28]:
# S
sbb_s_route_numbers = {
    "30392", "30661", "30698", "30794", "30821", "30823", "30825", "30827", "30829", "30831", "30833", "30835", "30837", "30839",
    "30841", "30843", "30845", "30847", "30849", "30851", "30853", "30855", "30857", "30859", "30861", "30863", "30865", "30867",
    "30869", "30871", "30873", "30949", "30967", "30985",
}
sbb_s_ranges = [
    (14303, 14398),
    (17001, 17099), (17113, 17394), (17525, 17568), (17906, 17943),
    (18011, 18020), (18087, 18093), (18220, 18993),
    (19010, 19293), (19416, 19693), (19921, 19980),
    (20024, 20580),
    (21016, 21395), (21912, 21997),
    (22015, 22175), (22960, 22991),
    (24500, 24697),
    (25100, 25792), (25905, 25993),
    (26018, 26075),
    (7606, 7743), (7819, 7892),
    (8416, 8999),
]
sbb_s_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "S")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_s_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_s_ranges))
)
data_chuuchuu.loc[sbb_s_mask, "operator"] = "SBB"
print(f"{sbb_s_mask.sum()} SBB S rows assigned 'SBB'")

# SN
sbb_sn_route_numbers = {
    "13702", "13704", "13707", "13709", "13710", "13711", "13712", "13713", "13714", "13715",
    "13716", "13717", "13746", "13748", "13750", "13751", "13752", "13753", "13754", "13755",
    "13756", "13757", "13760", "13762", "13763", "13764", "13765", "13766", "13767", "13769",
    "13770", "13771", "13772", "13773", "13774", "13775", "13776", "13777", "13780", "13781",
    "13782", "13783", "13784", "13785", "13786", "13787", "13790", "13791", "13792", "13793",
    "13794", "13795", "13796", "13797", "13811", "13812", "13813", "13814", "13815", "13816",
    "13818", "13830", "13831", "13832", "13834", "13835", "13836", "13837", "13838", "13839",
    "13840", "13841", "13842", "13843", "13844", "13845", "13846", "13847", "13849",
    "87795", "87797",
}
sbb_sn_mask = (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "SN") & data_chuuchuu["routeNumber_trim"].isin(sbb_sn_route_numbers)
data_chuuchuu.loc[sbb_sn_mask, "operator"] = "SBB"
print(f"{sbb_sn_mask.sum()} SBB SN rows assigned 'SBB'")

print("SBB PE / ICE / RB: no operator rule provided -- left as-is")

0 SBB S rows assigned 'SBB'
0 SBB SN rows assigned 'SBB'
SBB PE / ICE / RB: no operator rule provided -- left as-is


### OEBB agency

`CJX`/`D`/`ER`/`IR` are always OEBB in Austria. `EC`/`IC` split by `routeNumber` with an OEBB catch-all. `Os` is treated as exclusively Ceske Drahy. `R`/`REX`/`S` are OEBB in Austria except for a few excluded route numbers/ranges (run by other, smaller operators per the observed counts). `RB`/`UEX` have no rule. `WB` is always Westbahn.

In [29]:
oebb_always_mask = data_chuuchuu["routeType"].isin(["CJX", "D", "ER", "IR"]) & (data_chuuchuu["country"] == "Austria")
data_chuuchuu.loc[oebb_always_mask, "operator"] = "OEBB"
print(f"{oebb_always_mask.sum()} CJX/D/ER/IR rows in Austria assigned 'OEBB'")

# EC
oebb_ec_base = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "EC")

oebb_ec_oebb_numbers = {
    "100", "102", "106", "114", "140", "141", "142", "143", "144", "145", "146", "147", "148", "149", "164",
    "202", "204", "206", "212", "214", "290", "337", "340", "341", "342", "343", "462", "463", "70", "71", "78", "79",
}
oebb_ec_db_numbers = {"80", "81", "94", "96", "98", "115", "190", "192", "194", "196", "198", "213", "1281"}
oebb_ec_sbb_numbers = {"95", "97", "99", "163", "191", "193", "195", "197", "199"}
oebb_ec_pkp_numbers = {"101", "103", "107", "203", "205", "207"}

data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_oebb_numbers), "operator"] = "OEBB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_db_numbers), "operator"] = "DB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_sbb_numbers), "operator"] = "SBB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_pkp_numbers), "operator"] = "PKP Intercity"

oebb_ec_listed_numbers = oebb_ec_oebb_numbers | oebb_ec_db_numbers | oebb_ec_sbb_numbers | oebb_ec_pkp_numbers
oebb_ec_catchall = oebb_ec_base & ~data_chuuchuu["routeNumber_trim"].isin(oebb_ec_listed_numbers)
data_chuuchuu.loc[oebb_ec_catchall, "operator"] = "OEBB"
print(f"{oebb_ec_base.sum()} OEBB EC rows in Austria processed ({oebb_ec_catchall.sum()} via the OEBB catch-all)")

0 CJX/D/ER/IR rows in Austria assigned 'OEBB'
0 OEBB EC rows in Austria processed (0 via the OEBB catch-all)


In [30]:
# IC
oebb_ic_base = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "IC")

oebb_ic_db_numbers = {"406", "416"}
oebb_ic_pkp_numbers = {"207", "417"}

data_chuuchuu.loc[oebb_ic_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ic_db_numbers), "operator"] = "DB"
data_chuuchuu.loc[oebb_ic_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ic_pkp_numbers), "operator"] = "PKP Intercity"

oebb_ic_listed_numbers = oebb_ic_db_numbers | oebb_ic_pkp_numbers
oebb_ic_catchall = oebb_ic_base & ~data_chuuchuu["routeNumber_trim"].isin(oebb_ic_listed_numbers)
data_chuuchuu.loc[oebb_ic_catchall, "operator"] = "OEBB"
print(f"{oebb_ic_base.sum()} OEBB IC rows in Austria processed ({oebb_ic_catchall.sum()} via the OEBB catch-all)")

# Os -- treated as exclusively Ceske Drahy per the instructions (scoped to the OEBB agency, matching where this was observed)
os_mask = (data_chuuchuu["agency"] == "OEBB") & (data_chuuchuu["routeType"] == "Os")
data_chuuchuu.loc[os_mask, "operator"] = "Ceske Drahy"
print(f"{os_mask.sum()} OEBB 'Os' rows assigned 'Ceske Drahy'")

0 OEBB IC rows in Austria processed (0 via the OEBB catch-all)
0 OEBB 'Os' rows assigned 'Ceske Drahy'


In [31]:
# R
oebb_r_excluded_numbers = {"7807", "1826", "1828"}
oebb_r_mask = (
    (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "R")
    & ~data_chuuchuu["routeNumber_trim"].isin(oebb_r_excluded_numbers)
    & ~data_chuuchuu["routeNumber_int"].between(8000, 8200)
)
data_chuuchuu.loc[oebb_r_mask, "operator"] = "OEBB"
print(f"{oebb_r_mask.sum()} OEBB R rows in Austria assigned 'OEBB'")
print("OEBB RB: no operator rule provided -- left as-is")

# REX
oebb_rex_excluded = (
    (data_chuuchuu["routeNumber_trim"] == "5572")
    | data_chuuchuu["routeNumber_int"].between(8450, 8600)
    | data_chuuchuu["routeNumber_int"].between(7600, 7900)
)
oebb_rex_mask = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "REX") & ~oebb_rex_excluded
data_chuuchuu.loc[oebb_rex_mask, "operator"] = "OEBB"
print(f"{oebb_rex_mask.sum()} OEBB REX rows in Austria assigned 'OEBB'")

0 OEBB R rows in Austria assigned 'OEBB'
OEBB RB: no operator rule provided -- left as-is
0 OEBB REX rows in Austria assigned 'OEBB'


In [32]:
# S
oebb_s_excluded_numbers = {
    "5556", "5562", "5564", "5568", "5572", "5574", "5578", "5580", "5584", "5586",
    "5590", "5592", "5596", "5602", "5606", "5694", "25844", "25846",
    "25883", "25885", "25887", "25889", "25891", "25893", "25895", "25897",
}
oebb_s_excluded = (
    data_chuuchuu["routeNumber_int"].between(4350, 4378)
    | data_chuuchuu["routeNumber_int"].between(7350, 7388)
    | data_chuuchuu["routeNumber_int"].between(8000, 8544)
    | data_chuuchuu["routeNumber_trim"].isin(oebb_s_excluded_numbers)
)
oebb_s_mask = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "S") & ~oebb_s_excluded
data_chuuchuu.loc[oebb_s_mask, "operator"] = "OEBB"
print(f"{oebb_s_mask.sum()} OEBB S rows in Austria assigned 'OEBB'")
print("OEBB UEX: no operator rule provided -- left as-is")

# WB
wb_mask = data_chuuchuu["routeType"] == "WB"
data_chuuchuu.loc[wb_mask, "operator"] = "Westbahn"
print(f"{wb_mask.sum()} WB rows assigned 'Westbahn'")

0 OEBB S rows in Austria assigned 'OEBB'
OEBB UEX: no operator rule provided -- left as-is
0 WB rows assigned 'Westbahn'


### DB agency

`EC`/`ECE` split by `routeNumber`, each with its own catch-all label (`EC`'s catch-all is `'DB'`, `ECE`'s is the more specific `'DB Fernverkehr AG'` -- kept exactly as given rather than harmonized). `GV` is always Govolta. `IC` is split by `routeNumber` too, with `'DB Fernverkehr AG'` as its catch-all.

In [33]:
# EC
db_ec_base = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "EC")

db_ec_pkp_numbers = {
    "231", "247", "249", "41", "43", "431", "45", "47", "49", "55", "57", "59",
    "230", "246", "248", "40", "42", "430", "44", "46", "48", "54", "56", "58",
}
db_ec_sbb_numbers = {"150", "191", "193", "195", "197", "199", "95", "97", "99", "458", "459"}
db_ec_oebb_numbers = {"213", "115", "114", "212", "290"}

data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_pkp_numbers), "operator"] = "PKP Intercity"
data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_sbb_numbers), "operator"] = "SBB"
data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_oebb_numbers), "operator"] = "OEBB"

db_ec_listed_numbers = db_ec_pkp_numbers | db_ec_sbb_numbers | db_ec_oebb_numbers
db_ec_catchall = db_ec_base & ~data_chuuchuu["routeNumber_trim"].isin(db_ec_listed_numbers)
data_chuuchuu.loc[db_ec_catchall, "operator"] = "DB"
print(f"{db_ec_base.sum()} DB EC rows processed ({db_ec_catchall.sum()} via the 'DB' catch-all)")

# ECE
db_ece_base = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "ECE")
db_ece_sbb_numbers = {"190", "192", "194", "196", "198", "94", "96", "98", "151"}

data_chuuchuu.loc[db_ece_base & data_chuuchuu["routeNumber_trim"].isin(db_ece_sbb_numbers), "operator"] = "SBB"
db_ece_catchall = db_ece_base & ~data_chuuchuu["routeNumber_trim"].isin(db_ece_sbb_numbers)
data_chuuchuu.loc[db_ece_catchall, "operator"] = "DB Fernverkehr AG"
print(f"{db_ece_base.sum()} DB ECE rows processed ({db_ece_catchall.sum()} via the 'DB Fernverkehr AG' catch-all)")

# GV
gv_mask = data_chuuchuu["routeType"] == "GV"
data_chuuchuu.loc[gv_mask, "operator"] = "Govolta"
print(f"{gv_mask.sum()} GV rows assigned 'Govolta'")

# IC
db_ic_base = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "IC")

db_ic_sbb_numbers = {
    "180", "182", "184", "186", "188", "280", "282", "284", "380", "388", "480", "482", "484", "486", "488",
    "1082", "1180", "181", "183", "185", "187", "281", "283", "285", "389", "1089",
}
db_ic_pkp_numbers = {"132", "134", "407", "417"}
db_ic_oebb_numbers = {"406", "416"}
db_ic_dsb_mask = data_chuuchuu["routeNumber_int"].between(5751, 5767)

data_chuuchuu.loc[db_ic_base & data_chuuchuu["routeNumber_trim"].isin(db_ic_sbb_numbers), "operator"] = "SBB"
data_chuuchuu.loc[db_ic_base & data_chuuchuu["routeNumber_trim"].isin(db_ic_pkp_numbers), "operator"] = "PKP Intercity"
data_chuuchuu.loc[db_ic_base & data_chuuchuu["routeNumber_trim"].isin(db_ic_oebb_numbers), "operator"] = "OEBB"
data_chuuchuu.loc[db_ic_base & db_ic_dsb_mask, "operator"] = "DSB"

db_ic_listed_numbers = db_ic_sbb_numbers | db_ic_pkp_numbers | db_ic_oebb_numbers
db_ic_catchall = db_ic_base & ~data_chuuchuu["routeNumber_trim"].isin(db_ic_listed_numbers) & ~db_ic_dsb_mask
data_chuuchuu.loc[db_ic_catchall, "operator"] = "DB Fernverkehr AG"
print(f"{db_ic_base.sum()} DB IC rows processed ({db_ic_catchall.sum()} via the 'DB Fernverkehr AG' catch-all)")

0 DB EC rows processed (0 via the 'DB' catch-all)
0 DB ECE rows processed (0 via the 'DB Fernverkehr AG' catch-all)
0 GV rows assigned 'Govolta'
0 DB IC rows processed (0 via the 'DB Fernverkehr AG' catch-all)


### GTFSDE agency

`EC`/`ECE`/`EN` are intentionally left unassigned here -- they duplicate DB's own reporting of the same trains (already handled in the DB agency section above), so assigning them again would be redundant. `FEX`/`MEX`/`NRB`/`RB`/`RE`/`RS`/`U` are split by `routeNumber` (and, for `RB`/`RE`, also by `deutscheBahnStopId`, since routeNumber ranges alone overlap too much between DB regional entities and Arverio in those two categories).

In [34]:
# FEX
gtfsde_fex_mask = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "FEX") & data_chuuchuu["routeNumber_int"].between(21800, 21969)
data_chuuchuu.loc[gtfsde_fex_mask, "operator"] = "DB"
print(f"{gtfsde_fex_mask.sum()} GTFSDE FEX rows assigned 'DB'")

# MEX
gtfsde_mex_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "MEX")
mex_db_ranges = [(17500, 17570), (19200, 19399), (19500, 19699)]
mex_arverio_ranges = [(19100, 19199), (19400, 19499)]

mex_db_mask = gtfsde_mex_base & in_ranges(data_chuuchuu["routeNumber_int"], mex_db_ranges)
mex_arverio_mask = gtfsde_mex_base & in_ranges(data_chuuchuu["routeNumber_int"], mex_arverio_ranges)
data_chuuchuu.loc[mex_db_mask, "operator"] = "DB"
data_chuuchuu.loc[mex_arverio_mask, "operator"] = "Arverio"
print(f"{mex_db_mask.sum()} GTFSDE MEX rows assigned 'DB', {mex_arverio_mask.sum()} assigned 'Arverio'")

# NRB -- all of it is DB RegioNetz
gtfsde_nrb_mask = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "NRB")
data_chuuchuu.loc[gtfsde_nrb_mask, "operator"] = "DB"
print(f"{gtfsde_nrb_mask.sum()} GTFSDE NRB rows assigned 'DB'")

# RB -- routeNumber ranges alone overlap too much between operators, so DB additionally requires
# a specific set of deutscheBahnStopId values (the stops actually served by that DB regional entity)
gtfsde_rb_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "RB")

rb_arverio_ranges = [(57000, 57344), (78901, 78975)]
rb_arverio_mask = gtfsde_rb_base & in_ranges(data_chuuchuu["routeNumber_int"], rb_arverio_ranges)
data_chuuchuu.loc[rb_arverio_mask, "operator"] = "Arverio"

rb_db_stop_ids = ['331064', '360304', '5100082', '5100083', '5100096', '5100222', '5101281', '5102886', '5189954', '5193610', '8010016', '8010018', '8010036', '8010041', '8010051', '8010053', '8010066', '8010069', '8010072', '8010073', '8010079', '8010089', '8010093', '8010099', '8010100', '8010103', '8010113', '8010176', '8010183', '8010193', '8010215', '8010255', '8010279', '8010280', '8010285', '8010300', '8010304', '8010308', '8010322', '8010324', '8010327', '8010338', '8010355', '8010357', '8010373', '8010377', '8010381', '8010389', '8010392', '8010395', '8010396', '8010403', '8010404', '8010405', '8010406', '8011031', '8011078', '8011093', '8011098', '8011102', '8011108', '8011109', '8011114', '8011140', '8011155', '8011160', '8011162', '8011167', '8011179', '8011188', '8011201', '8011270', '8011286', '8011306', '8011318', '8011319', '8011320', '8011334', '8011340', '8011414', '8011419', '8011421', '8011425', '8011471', '8011540', '8011542', '8011563', '8011667', '8011695', '8011729', '8011735', '8011749', '8011778', '8011797', '8011889', '8011901', '8011944', '8011945', '8011991', '8011992', '8011995', '8012006', '8012065', '8012084', '8012086', '8012089', '8012096', '8012108', '8012127', '8012169', '8012215', '8012253', '8012315', '8012316', '8012329', '8012341', '8012377', '8012445', '8012469', '8012479', '8012482', '8012503', '8012582', '8012583', '8012584', '8012609', '8012617', '8012621', '8012650', '8012666', '8012681', '8012713', '8012729', '8012785', '8012806', '8012818', '8012819', '8012840', '8012841', '8012892', '8012903', '8012934', '8012941', '8012962', '8012963', '8013021', '8013040', '8013105', '8013106', '8013132', '8013133', '8013160', '8013161', '8013183', '8013185', '8013267', '8013272', '8013305', '8013339', '8013340', '8013341', '8013350', '8013368', '8013385', '8013406', '8013470', '8013475', '8013481', '8013483', '8013487', '8013489', '8013490', '8017349', '8079084', '8079604', '8079629', '8080170', '8080190', '8080260', '8080370', '8080710', '8081220', '8081688', '8087026', '8087027', '936003']
rb_db_ranges = [
    (5383, 5395), (5830, 5841), (18100, 18166), (18238, 18275), (18300, 18341),
    (93250, 93252), (93268, 93270), (94740, 94775), (18700, 18737), (18748, 18749), (18800, 18869),
    (13030, 13037), (18000, 18048), (18080, 18084), (18550, 18599),
    (13100, 13149), (13221, 13269), (18740, 18745), (18870, 18899), (18345, 18372), (18420, 18445),
]
rb_db_single_numbers = {"18451", "18480", "3648", "18086"}

rb_db_mask = (
    gtfsde_rb_base
    & data_chuuchuu["deutscheBahnStopId"].isin(rb_db_stop_ids)
    & (in_ranges(data_chuuchuu["routeNumber_int"], rb_db_ranges) | data_chuuchuu["routeNumber_trim"].isin(rb_db_single_numbers))
)
data_chuuchuu.loc[rb_db_mask, "operator"] = "DB"
print(f"{rb_arverio_mask.sum()} GTFSDE RB rows assigned 'Arverio', {rb_db_mask.sum()} assigned 'DB'")

# RE -- same stopId + routeNumber approach as RB
gtfsde_re_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "RE")

re_arverio_ranges = [(57006, 57347), (78900, 78984)]
re_arverio_mask = gtfsde_re_base & in_ranges(data_chuuchuu["routeNumber_int"], re_arverio_ranges)
data_chuuchuu.loc[re_arverio_mask, "operator"] = "Arverio"

re_db_stop_ids = ['8000042', '8000124', '8000131', '8000156', '8000189', '8000191', '8000218', '8000229', '8000236', '8000244', '8000264', '8000265', '8000275', '8000295', '8000323', '8000369', '8000373', '8000383', '8000423', '8000471', '8000599', '8000649', '8000668', '8000681', '8000736', '8001132', '8001366', '8001618', '8001707', '8001883', '8002021', '8002137', '8002342', '8002380', '8002632', '8002685', '8002883', '8002931', '8003101', '8003235', '8003726', '8003759', '8003932', '8004094', '8004095', '8004215', '8004219', '8004577', '8004658', '8005013', '8005077', '8005229', '8005494', '8005578', '8005592', '8005714', '8005736', '8006083', '8006137', '8006661', '8070097', '8700271', '8700439']
re_db_ranges = [(38761, 38770), (38784, 38799), (4280, 4299), (13324, 13334), (86381, 86391), (88824, 88873)]
re_db_single_numbers = {"38632", "38710", "38112"}

re_db_mask = (
    gtfsde_re_base
    & data_chuuchuu["deutscheBahnStopId"].isin(re_db_stop_ids)
    & (in_ranges(data_chuuchuu["routeNumber_int"], re_db_ranges) | data_chuuchuu["routeNumber_trim"].isin(re_db_single_numbers))
)
data_chuuchuu.loc[re_db_mask, "operator"] = "DB"
print(f"{re_arverio_mask.sum()} GTFSDE RE rows assigned 'Arverio', {re_db_mask.sum()} assigned 'DB'")

# RS
gtfsde_rs_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "RS")
rs_db_mask = gtfsde_rs_base & in_ranges(data_chuuchuu["routeNumber_int"], [(32600, 32677), (57440, 57798)])
data_chuuchuu.loc[rs_db_mask, "operator"] = "DB"
print(f"{rs_db_mask.sum()} GTFSDE RS rows assigned 'DB'")

# U
gtfsde_u_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "U")
u_db_mask = gtfsde_u_base & (data_chuuchuu["routeNumber_int"].between(28000, 28014) | (data_chuuchuu["routeNumber_int"] == 28023))
data_chuuchuu.loc[u_db_mask, "operator"] = "DB"
print(f"{u_db_mask.sum()} GTFSDE U rows assigned 'DB'")

print("GTFSDE EC/ECE/EN: intentionally left unassigned -- duplicates of DB's own reporting")

0 GTFSDE FEX rows assigned 'DB'
0 GTFSDE MEX rows assigned 'DB', 0 assigned 'Arverio'
0 GTFSDE NRB rows assigned 'DB'
0 GTFSDE RB rows assigned 'Arverio', 0 assigned 'DB'
0 GTFSDE RE rows assigned 'Arverio', 0 assigned 'DB'
0 GTFSDE RS rows assigned 'DB'
0 GTFSDE U rows assigned 'DB'
GTFSDE EC/ECE/EN: intentionally left unassigned -- duplicates of DB's own reporting


### OEBB agency: corrections from a later re-investigation

These come from the data provider re-examining OEBB's own EC/EN/IC reporting specifically, and **intentionally override** the earlier OEBB EC section above and the generic `EN` night-train assignment for these particular `(agency=OEBB, routeType, routeNumber)` combinations -- e.g. some `EN` route numbers previously assigned `HZ`/`MAV` by the generic table turn out to actually be `OEBB`/`PKP Intercity` when reported under the OEBB agency specifically.

Note: the source instructions' heading says "agency OEBB and SBB", but the actual condition only filters `agency = 'OEBB'` -- implemented exactly as the condition states, not the heading.

In [35]:
oebb_correction_db_ec_numbers = {"115", "1281", "190", "192", "194", "196", "198", "213", "80", "81", "94", "96", "98"}
oebb_correction_db_ic_numbers = {"406", "416"}
oebb_correction_db_mask = (
    (data_chuuchuu["agency"] == "OEBB")
    & (
        ((data_chuuchuu["routeType"] == "EC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_db_ec_numbers))
        | ((data_chuuchuu["routeType"] == "IC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_db_ic_numbers))
    )
)
data_chuuchuu.loc[oebb_correction_db_mask, "operator"] = "DB"

oebb_correction_oebb_ec_numbers = {
    "100", "102", "106", "114", "140", "141", "142", "143", "144", "145", "146", "147", "148", "149", "164",
    "202", "204", "206", "212", "214", "290", "337", "340", "341", "342", "343", "462", "463", "70", "71", "78", "79",
}
oebb_correction_oebb_en_numbers = {"40237", "40414", "40462", "40465", "40467", "414", "50237"}
oebb_correction_oebb_ic_numbers = {
    "1110", "1111", "1112", "1113", "1115", "1118", "1119", "1135", "1136", "1138", "1142", "1143", "1151", "1244", "1249",
    "350", "351", "354", "407", "460", "532", "533", "534", "535", "536", "537", "538", "540", "541", "542", "543", "544",
    "545", "546", "547", "548", "549", "558", "559", "640", "641", "642", "643", "644", "645", "646", "647", "648", "649",
    "651", "740", "741", "742", "743", "744", "745", "746", "747", "748", "749", "756", "759", "790", "791", "792", "793",
    "794", "795", "796", "797", "798", "799", "840", "841", "842", "843", "847", "848", "850", "890", "891", "896", "897",
    "898", "899",
}
oebb_correction_oebb_mask = (
    (data_chuuchuu["agency"] == "OEBB")
    & (
        ((data_chuuchuu["routeType"] == "EC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_oebb_ec_numbers))
        | ((data_chuuchuu["routeType"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_oebb_en_numbers))
        | ((data_chuuchuu["routeType"] == "IC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_oebb_ic_numbers))
    )
)
data_chuuchuu.loc[oebb_correction_oebb_mask, "operator"] = "OEBB"

oebb_correction_pkp_ec_numbers = {"101", "103", "107", "203", "205", "207"}
oebb_correction_pkp_en_numbers = {"40406", "40407", "40416", "40417"}
oebb_correction_pkp_ic_numbers = {"207", "417"}
oebb_correction_pkp_mask = (
    (data_chuuchuu["agency"] == "OEBB")
    & (
        ((data_chuuchuu["routeType"] == "EC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_pkp_ec_numbers))
        | ((data_chuuchuu["routeType"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_pkp_en_numbers))
        | ((data_chuuchuu["routeType"] == "IC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_pkp_ic_numbers))
    )
)
data_chuuchuu.loc[oebb_correction_pkp_mask, "operator"] = "PKP Intercity"

oebb_correction_sbb_ec_numbers = {"163", "191", "193", "195", "197", "199", "95", "97", "99"}
oebb_correction_sbb_mask = (
    (data_chuuchuu["agency"] == "OEBB") & (data_chuuchuu["routeType"] == "EC")
    & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_sbb_ec_numbers)
)
data_chuuchuu.loc[oebb_correction_sbb_mask, "operator"] = "SBB"

print(f"OEBB corrections applied: {oebb_correction_db_mask.sum()} -> DB, {oebb_correction_oebb_mask.sum()} -> OEBB, {oebb_correction_pkp_mask.sum()} -> PKP Intercity, {oebb_correction_sbb_mask.sum()} -> SBB")

OEBB corrections applied: 0 -> DB, 0 -> OEBB, 0 -> PKP Intercity, 0 -> SBB


### Fallback: assume the local national operator where nothing else assigned one

For a handful of remaining unassigned combinations, the data provider suggests falling back to "whichever operator is national to the country the stop is in" -- applied only where `operator` is still null, so it never overrides anything assigned above.

In [36]:
# DB agency, EN routeType: fill in the leftover (e.g. the "no info" route numbers listed in the
# night train section) by country
db_en_fallback_map = {"Germany": "DB", "Austria": "OEBB", "Switzerland": "SBB", "Poland": "PKP Intercity", "Czech Republic": "CD"}
db_en_fallback_mask = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "EN") & data_chuuchuu["operator"].isna()
data_chuuchuu.loc[db_en_fallback_mask, "operator"] = data_chuuchuu.loc[db_en_fallback_mask, "country"].map(db_en_fallback_map)
print(f"{db_en_fallback_mask.sum()} DB EN rows with no operator yet had a country-based fallback attempted")

# OEBB agency, EN/EC/IC/ECE routeTypes -- the source instructions' heading mentions "OEBB and SBB"
# but the actual condition only filters agency = 'OEBB'; implemented as the condition states
oebb_fallback_map = {
    "Germany": "DB", "Austria": "OEBB", "Switzerland": "SBB", "Poland": "PKP Intercity",
    "Czech Republic": "CD", "Italy": "Trenitalia",
}
oebb_fallback_mask = (
    (data_chuuchuu["agency"] == "OEBB") & data_chuuchuu["routeType"].isin(["EN", "EC", "IC", "ECE"])
    & data_chuuchuu["operator"].isna()
)
data_chuuchuu.loc[oebb_fallback_mask, "operator"] = data_chuuchuu.loc[oebb_fallback_mask, "country"].map(oebb_fallback_map)
print(f"{oebb_fallback_mask.sum()} OEBB EN/EC/IC/ECE rows with no operator yet had a country-based fallback attempted")

# IT agency, EC routeType: not covered by the earlier "easy operator cases" IT rule (which didn't
# include EC) -- always Trenitalia when the stop is in Italy or France
it_ec_fallback_mask = (
    (data_chuuchuu["agency"] == "IT") & (data_chuuchuu["routeType"] == "EC") & data_chuuchuu["operator"].isna()
    & data_chuuchuu["country"].isin(["Italy", "France"])
)
data_chuuchuu.loc[it_ec_fallback_mask, "operator"] = "Trenitalia"
print(f"{it_ec_fallback_mask.sum()} IT EC rows with no operator yet assigned 'Trenitalia' (Italy/France)")

0 DB EN rows with no operator yet had a country-based fallback attempted
0 OEBB EN/EC/IC/ECE rows with no operator yet had a country-based fallback attempted
0 IT EC rows with no operator yet assigned 'Trenitalia' (Italy/France)


In [37]:
data_chuuchuu["normalized_operator"] = data_chuuchuu["operator"]

db_regio_mask = data_chuuchuu["operator"].str.contains("DB Regio", case=False, na=False)
data_chuuchuu.loc[db_regio_mask, "normalized_operator"] = "DB Regio"
print(f"{db_regio_mask.sum()} rows with operator containing 'DB Regio' assigned 'DB Regio'")

0 rows with operator containing 'DB Regio' assigned 'DB Regio'


### Cleanup & summary

In [38]:
data_chuuchuu = data_chuuchuu.drop(columns=["routeType_norm", "agency_norm", "routeNumber_trim", "routeNumber_int"])

print(f"{data_chuuchuu['normalized_operator'].isna().sum()} rows ({data_chuuchuu['normalized_operator'].isna().mean() * 100:.2f}%) still have no normalized_operator")
print()
data_chuuchuu["normalized_operator"].value_counts(dropna=False).head(30)

0 rows (0.00%) still have no normalized_operator



normalized_operator
SNCF                                       6423335
SNCF VOYAGEURS                              337967
Eurostar                                    153926
Deutsche Bahn                                62093
Trenitalia                                   24854
SNCF Voyageurs LO                            21041
OCEdefault                                   12469
Conseil Régional Auvergne - Rhône-Alpes       1727
SNCF Voyageurs EA                              718
SNCF Voyageurs SA                               79
Name: count, dtype: int64

## Step 4 -- Cancellation status resolution

Resolve a best-effort cancellation status for every stop, filling in `arrivalCancelled`/`departureCancelled` where they're null but can be inferred from other columns.

In [39]:
arr_cancelled_t = (data_chuuchuu["arrivalCancelled"] == "t").mean() * 100
arr_cancelled_f = (data_chuuchuu["arrivalCancelled"] == "f").mean() * 100
dep_cancelled_t = (data_chuuchuu["departureCancelled"] == "t").mean() * 100
dep_cancelled_f = (data_chuuchuu["departureCancelled"] == "f").mean() * 100

print(f"arrivalCancelled: {arr_cancelled_t:.2f}% 't', {arr_cancelled_f:.2f}% 'f', {100 - arr_cancelled_t - arr_cancelled_f:.2f}% no data")
print(f"departureCancelled: {dep_cancelled_t:.2f}% 't', {dep_cancelled_f:.2f}% 'f', {100 - dep_cancelled_t - dep_cancelled_f:.2f}% no data")

arrivalCancelled: 1.67% 't', 95.64% 'f', 2.69% no data
departureCancelled: 1.67% 't', 95.64% 'f', 2.69% no data


### Recovering a cancellation status for rows where `arrivalCancelled` is null

Per the terminology doc: *"If null and a delay was recorded, you can assume the arrival was not cancelled."* But most null-`arrivalCancelled` rows turn out to be departure stops -- which have no arrival to record in the first place, so `arrival` itself is uninformative here. To be sure the train wasn't cancelled, we instead check whether an effective `departure` timestamp exists: if it does, the train was there and left, so the (inapplicable) arrival wasn't a cancellation.

In [40]:
arr_null_cancelled = data_chuuchuu["arrivalCancelled"].isna()

# among the rows with no explicit flag, an effective departure timestamp means it wasn't cancelled
arr_inferred_not_cancelled = arr_null_cancelled & data_chuuchuu["departure"].notna()

# still no way to know -- neither an explicit flag nor an effective departure to infer from
arr_unreliable = arr_null_cancelled & data_chuuchuu["departure"].isna()

n = len(data_chuuchuu)
print(f"Explicit arrivalCancelled value (t/f): {(~arr_null_cancelled).sum() / n * 100:.2f}%")
print(f"Null but inferred not cancelled (departure is not null): {arr_inferred_not_cancelled.sum() / n * 100:.2f}%")
print(f"No reliable cancellation data at all: {arr_unreliable.sum() / n * 100:.2f}%")

data_chuuchuu["arrivalCancelled_resolved"] = data_chuuchuu["arrivalCancelled"]
data_chuuchuu.loc[arr_inferred_not_cancelled, "arrivalCancelled_resolved"] = "f"

data_chuuchuu["arrivalCancelled_resolved"].value_counts(dropna=False)

Explicit arrivalCancelled value (t/f): 97.31%
Null but inferred not cancelled (departure is not null): 2.69%
No reliable cancellation data at all: 0.00%


arrivalCancelled_resolved
f    6920658
t     117551
Name: count, dtype: int64

### Recovering a cancellation status for rows where `departureCancelled` is null

Same idea for departures, with two inference steps:

1. If the stop is the journey's `terminus`, there's no onward departure to make -- a null `departureCancelled` there just means the field doesn't apply, not that it's unknown.
2. Otherwise, if the *next* stop on the same journey has a confirmed arrival (`arrivalCancelled_resolved == "f"`), the train must have departed the current stop to get there.

In [41]:
dep_null_cancelled = data_chuuchuu["departureCancelled"].isna()

# among the rows with no explicit flag, being at the terminus means there's no departure to resolve
dep_inferred_not_cancelled = dep_null_cancelled & (data_chuuchuu["depart_terminus"] == "terminus")
dep_unreliable = dep_null_cancelled & (data_chuuchuu["depart_terminus"] != "terminus")

n = len(data_chuuchuu)
print(f"Explicit departureCancelled value (t/f): {(~dep_null_cancelled).sum() / n * 100:.2f}%")
print(f"Null but inferred not cancelled (stop is the terminus): {dep_inferred_not_cancelled.sum() / n * 100:.2f}%")
print(f"No reliable cancellation data yet: {dep_unreliable.sum() / n * 100:.2f}%")

data_chuuchuu["departureCancelled_resolved"] = data_chuuchuu["departureCancelled"]
data_chuuchuu.loc[dep_inferred_not_cancelled, "departureCancelled_resolved"] = "f"

data_chuuchuu["departureCancelled_resolved"].value_counts(dropna=False)

Explicit departureCancelled value (t/f): 97.31%
Null but inferred not cancelled (stop is the terminus): 2.65%
No reliable cancellation data yet: 0.03%


departureCancelled_resolved
f      6918315
t       117551
NaN       2343
Name: count, dtype: int64

### Recovering more `departureCancelled` via the next stop's arrival

Some intermediate stops have a confirmed arrival (`arrivalCancelled_resolved == "f"`, i.e. the train reached the station) but a null `departureCancelled` -- so we don't yet know if it left. If the *next* stop on the same journey also has a confirmed arrival, the train must have departed the current stop to get there, so `departureCancelled_resolved` can be set to `"f"`.

This only targets non-terminus stops -- a terminus stop has no onward departure to resolve, and is already handled by the terminus-based inference above, so there's no overlap between the two.

In [42]:
# sort each journey's stops chronologically, then look one row ahead within the same journey
# (groupby + shift is vectorized -- no explicit loop over rows)
sorted_by_journey = data_chuuchuu.sort_values(["journey_id", "sort_time"])
next_arrival_resolved = sorted_by_journey.groupby("journey_id")["arrivalCancelled_resolved"].shift(-1)
next_stop_arrived = (next_arrival_resolved == "f").reindex(data_chuuchuu.index)

dep_inferred_from_next_stop = (
    (data_chuuchuu["depart_terminus"] != "terminus")
    & data_chuuchuu["departureCancelled"].isna()
    & (data_chuuchuu["arrivalCancelled_resolved"] == "f")
    & next_stop_arrived
)

print(f"{dep_inferred_from_next_stop.sum()} of {dep_unreliable.sum()} previously unreliable departureCancelled rows resolved via next-stop arrival")

data_chuuchuu.loc[dep_inferred_from_next_stop, "departureCancelled_resolved"] = "f"

data_chuuchuu["departureCancelled_resolved"].value_counts(dropna=False)

2317 of 2343 previously unreliable departureCancelled rows resolved via next-stop arrival


departureCancelled_resolved
f      6920632
t       117551
NaN         26
Name: count, dtype: int64

## Step 5 -- Per-train summary & export

Everything above works at the stop level. This step collapses each `journey_id` down to a single per-train record, then aggregates by year / operator / route type into the final summary.

We reuse the same exclusions from step 2: a journey with a repeated stop id (`is_ambiguous_trip`) can't be ranked into depart/intermediate/terminus, and a `journey_verificator` collision flagged `likely_true_duplicate` is almost certainly the same physical train reported twice by two agencies -- keeping both would double-count it.

**Cancellation** is judged across every stop of the journey (using `arrivalCancelled_resolved` from step 4):
- a journey is only classified if every one of its stops has a known cancellation status -- a single unresolved stop could be hiding either outcome, so we don't guess
- **full cancellation**: all stops cancelled
- **partial cancellation**: some, but not all, stops cancelled

**Terminus outcome** (`cancelled_terminus`) looks specifically at the journey's `terminus` stop: was it cancelled, not cancelled, or unresolved (no terminus row could be identified for that journey)? `arrived_at_terminus` is true only when the terminus was resolved and confirmed not cancelled. Delay (>5min / >15min) is measured on that same terminus row's `arrivalDelay` -- a cancelled or unresolved terminus has no delay to measure, so it's excluded from both delay counts rather than counted as on-time.

In [43]:
# this summary needs one row per train, not one row per stop -- exclude the same
# unreliable journeys flagged in step 2 (see markdown above)
clean = ~data_chuuchuu["is_ambiguous_trip"] & (data_chuuchuu["cross_agency_duplicate_confidence"] != "likely_true_duplicate")
print(f"{(~clean).sum()} rows excluded as an ambiguous trip or a likely cross-agency duplicate")

clean_data = data_chuuchuu[clean].copy()
clean_data["year"] = pd.to_datetime(clean_data["date"]).dt.year

46 rows excluded as an ambiguous trip or a likely cross-agency duplicate


In [44]:
stop_counts = clean_data.groupby("journey_id").agg(
    n_stops=("arrivalCancelled_resolved", "size"),
    n_known_cancellation=("arrivalCancelled_resolved", lambda s: s.notna().sum()),
    n_cancelled_stops=("arrivalCancelled_resolved", lambda s: (s == "t").sum()),
)

# only classify a journey's cancellation if every one of its stops has a known status
stop_counts["cancellation_reliable"] = stop_counts["n_known_cancellation"] == stop_counts["n_stops"]
stop_counts["is_fully_cancelled"] = stop_counts["cancellation_reliable"] & (stop_counts["n_cancelled_stops"] == stop_counts["n_stops"])
stop_counts["is_partially_cancelled"] = (
    stop_counts["cancellation_reliable"]
    & (stop_counts["n_cancelled_stops"] > 0)
    & (stop_counts["n_cancelled_stops"] < stop_counts["n_stops"])
)

print(f"{(~stop_counts['cancellation_reliable']).sum()} of {len(stop_counts)} journeys have no reliable cancellation status (excluded from full/partial counts)")
stop_counts[["is_fully_cancelled", "is_partially_cancelled"]].sum()

0 of 857791 journeys have no reliable cancellation status (excluded from full/partial counts)


is_fully_cancelled        11822
is_partially_cancelled     9714
dtype: int64

In [45]:
# journey attributes (operator, route type, year) are constant across a journey's stops by
# construction (journey_id == agency + routeType + routeNumber + date), so any stop can represent them
journey_attrs = clean_data.drop_duplicates("journey_id").set_index("journey_id")[
    ["agency", "normalized_operator", "routeType", "year", "journey_type"]
].rename(columns={"normalized_operator": "operator"})

terminus_rows = clean_data.loc[clean_data["depart_terminus"] == "terminus"].set_index("journey_id")
cancelled_terminus = terminus_rows["arrivalCancelled_resolved"].map({"t": True, "f": False})  # NaN where unresolved

journeys = journey_attrs.join(stop_counts)
journeys["cancelled_terminus"] = journeys.index.map(cancelled_terminus)
journeys["terminus_arrival_delay"] = journeys.index.map(terminus_rows["arrivalDelay"])

# arrived at terminus: the terminus stop was identified AND confirmed not cancelled (NaN --
# no terminus row could be resolved for that journey -- correctly evaluates to False here)
journeys["arrived_at_terminus"] = journeys["cancelled_terminus"] == False

DELAY_5MIN_SECONDS = 5 * 60
DELAY_15MIN_SECONDS = 15 * 60
journeys["delayed_5min"] = journeys["arrived_at_terminus"] & (journeys["terminus_arrival_delay"] > DELAY_5MIN_SECONDS)
journeys["delayed_15min"] = journeys["arrived_at_terminus"] & (journeys["terminus_arrival_delay"] > DELAY_15MIN_SECONDS)

# the groupby below needs journey_id as a regular column, not the index
journeys = journeys.reset_index()

print(f"{len(journeys)} journeys total")
print(f"{journeys['arrived_at_terminus'].sum()} arrived at terminus")
print(f"{journeys['delayed_5min'].sum()} delayed >5min, {journeys['delayed_15min'].sum()} delayed >15min (at terminus)")

journeys.head()

857791 journeys total
843040 arrived at terminus
95848 delayed >5min, 49188 delayed >15min (at terminus)


,journey_id,agency,operator,routeType,year,journey_type,n_stops,n_known_cancellation,n_cancelled_stops,cancellation_reliable,is_fully_cancelled,is_partially_cancelled,cancelled_terminus,terminus_arrival_delay,arrived_at_terminus,delayed_5min,delayed_15min
0,FR_INTERCITES_3604_2025-01-01,FR,SNCF,INTERCITES,2025,domestic,7,7,0,True,False,False,False,0.0,True,False,False
1,FR_TGV INOUI_8330_2025-01-01,FR,SNCF,TGV INOUI,2025,domestic,5,5,0,True,False,False,False,0.0,True,False,False
2,FR_TGV INOUI_2501_2025-01-01,FR,SNCF,TGV INOUI,2025,domestic,3,3,0,True,False,False,False,0.0,True,False,False
3,FR_INTERCITES_5950_2025-01-01,FR,SNCF,INTERCITES,2025,domestic,6,6,0,True,False,False,False,0.0,True,False,False
4,FR_TGV INOUI_2714_2025-01-01,FR,SNCF,TGV INOUI,2025,domestic,5,5,0,True,False,False,False,0.0,True,False,False


### Aggregating to Year / operator / Route type

Each row of `journeys` (built above) is already one train. `operator` is null wherever step 3 couldn't resolve one -- rather than silently dropping those trains from the groupby, they're kept under an explicit `UNKNOWN_OPERATOR` label so the totals in the summary still add up to the full `journeys` count.

Column definitions (see the per-train summary section above for how each is derived):
- **N total train** -- every train in scope, regardless of cancellation/delay/operator status
- **N cancelled train (full cancellation)** -- every stop of the train was cancelled
- **N cancelled train (partial cancellation)** -- some, but not all, stops were cancelled
- **N cancelled train (full + partial)** -- either of the above
- **N train delay (> 5 min) / (> 15 min)** -- arrived at the terminus, but later than the threshold
- **N train arriving at terminus** -- the terminus stop was reached (not cancelled), independent of delay

In [46]:
journeys["operator"] = journeys["operator"].fillna("UNKNOWN_OPERATOR")
print(f"{(journeys['operator'] == 'UNKNOWN_OPERATOR').mean() * 100:.2f}% of trains have no resolved operator")

summary = journeys.groupby(["year", "operator", "routeType"], dropna=False).agg(
    n_total_train=("journey_id", "size"),
    n_cancelled_full=("is_fully_cancelled", "sum"),
    n_cancelled_partial=("is_partially_cancelled", "sum"),
    n_delay_5min=("delayed_5min", "sum"),
    n_delay_15min=("delayed_15min", "sum"),
    n_arriving_terminus=("arrived_at_terminus", "sum"),
).reset_index()

summary["n_cancelled_full_or_partial"] = summary["n_cancelled_full"] + summary["n_cancelled_partial"]

assert summary["n_total_train"].sum() == len(journeys), "grouped total doesn't match the journey count -- a group key must be dropping nulls"

summary.head()

0.00% of trains have no resolved operator


,year,operator,routeType,n_total_train,n_cancelled_full,n_cancelled_partial,n_delay_5min,n_delay_15min,n_arriving_terminus,n_cancelled_full_or_partial
0,2025,Conseil Régional Auvergne - Rhône-Alpes,CAR,56,0,0,0,0,56,0
1,2025,Conseil Régional Auvergne - Rhône-Alpes,CTE,11,0,0,8,0,11,0
2,2025,Conseil Régional Auvergne - Rhône-Alpes,Car,176,0,29,0,0,176,29
3,2025,Deutsche Bahn,ICE,8289,13,207,3776,1886,8218,220
4,2025,Eurostar,EST,42687,1775,2646,10548,6311,40398,4421


### Exporting the summary as Excel

In [47]:
summary = summary.rename(columns={
    "year": "Year",
    "routeType": "Route type",
    "n_total_train": "N total train",
    "n_cancelled_full": "N cancelled train (full cancellation)",
    "n_cancelled_partial": "N cancelled train (partial cancellation)",
    "n_cancelled_full_or_partial": "N cancelled train (full + partial)",
    "n_delay_5min": "N train delay (> 5 min)",
    "n_delay_15min": "N train delay (> 15 min)",
    "n_arriving_terminus": "N train arriving at terminus",
})

column_order = [
    "Year", "operator", "Route type",
    "N total train",
    "N cancelled train (full cancellation)",
    "N cancelled train (partial cancellation)",
    "N cancelled train (full + partial)",
    "N train delay (> 5 min)",
    "N train delay (> 15 min)",
    "N train arriving at terminus",
]
summary = summary[column_order].sort_values(["Year", "operator", "Route type"])
summary

,Year,operator,Route type,N total train,N cancelled train (full cancellation),N cancelled train (partial cancellation),N cancelled train (full + partial),N train delay (> 5 min),N train delay (> 15 min),N train arriving at terminus
0,2025,Conseil Régional Auvergne - Rhône-Alpes,CAR,56,0,0,0,0,0,56
1,2025,Conseil Régional Auvergne - Rhône-Alpes,CTE,11,0,0,0,8,0,11
2,2025,Conseil Régional Auvergne - Rhône-Alpes,Car,176,0,29,29,0,0,176
3,2025,Deutsche Bahn,ICE,8289,13,207,220,3776,1886,8218
4,2025,Eurostar,EST,42687,1775,2646,4421,10548,6311,40398
5,2025,OCEdefault,CTE,1537,0,0,0,51,23,1537
6,2025,SNCF,CAR TER,13630,0,0,0,1211,233,13630
7,2025,SNCF,IC,3423,37,31,68,730,461,3375
8,2025,SNCF,ICN,504,46,12,58,26,18,453
9,2025,SNCF,INTERCITES,18995,218,226,444,4208,2723,18687


In [48]:
export_summary = input("Export summary to Excel? (y/n): ")

if export_summary.lower() == "y":
    summary_stats_dir = "summary_stats"
    os.makedirs(summary_stats_dir, exist_ok=True)

    summary_path = f"{summary_stats_dir}/chuuchuu_summary_{data_selection}.xlsx"
    summary.to_excel(summary_path, index=False)
    print(f"Saved to {summary_path}")

Saved to summary_stats/chuuchuu_summary_french.xlsx


### Exporting the enriched dataset

The Excel summary above is aggregated (one row per year / operator / route type). This step separately saves the full per-stop `data_chuuchuu` dataframe -- with all the columns added in steps 1-4 (`country`, `journey_id`, `journey_type`, `normalized_operator`, `arrivalCancelled_resolved` / `departureCancelled_resolved`) -- to `intermediate_outputs/` as a parquet file, so other analyses in the project can reuse the enriched per-stop data without re-running this notebook.

In [49]:
export_enriched = input("Export enriched dataset to intermediate_outputs/? (y/n): ")

if export_enriched.lower() == "y":
    intermediate_outputs_dir = "intermediate_outputs"
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    enriched_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_enriched.parquet"
    data_chuuchuu.to_parquet(enriched_path)
    print(f"Saved to {enriched_path}")

Saved to intermediate_outputs/data_chuuchuu_french_enriched.parquet
